# Penyelarasan Data Temporal — Berita Geopolitik & Kurs USD/IDR

Notebook ini menyelaraskan data berita geopolitik (`geopolitical_news.csv`, hasil `scraper.ipynb`) dengan data kurs harian USD/IDR dari Bank Indonesia (`jisdor_usd_idr.csv`, hasil `timeseries_jisdor.ipynb`) berdasarkan waktu terbit beritanya.

## 0. Import Library

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 80)

## 1. Latar Belakang Masalah

Dua sumber data yang kita punya punya karakteristik waktu yang berbeda:

- **Data kurs (JISDOR)** hanya tersedia di hari kerja pasar sehingga tidak ada nilai di akhir pekan atau hari libur nasional.
- **Data berita** bisa terbit kapan saja, 24 jam, termasuk akhir pekan dan hari libur.

Akibatnya, sebuah berita yang terbit di hari Sabtu, Minggu, hari libur, atau setelah jam tutup pasar pada hari kerja, tidak punya "pasangan" langsung di data kurs pada tanggal yang sama persis. Notebook ini menetapkan aturan logis untuk memetakan tiap berita ke hari trading yang paling relevan, tanpa membuang informasinya.

## Load Data

In [2]:
DATA_DIR = Path("../data")

kurs = pd.read_csv(DATA_DIR / "processed" / "jisdor_usd_idr.csv", parse_dates=["date"])
kurs = kurs.sort_values("date").reset_index(drop=True)

news = pd.read_csv(DATA_DIR / "processed" / "articles_clean.csv", parse_dates=["published_at"])  # sudah dibersihkan preprocessing.ipynb

print(f"Data kurs   : {len(kurs)} baris, {kurs['date'].min().date()} s.d. {kurs['date'].max().date()}")
print(f"Data berita : {len(news)} baris, {news['published_at'].min()} s.d. {news['published_at'].max()}")

kurs.head()

Data kurs   : 1202 baris, 2021-09-01 s.d. 2026-09-01
Data berita : 1719 baris, 2021-09-01 07:00:00 s.d. 2026-09-09 03:37:00


,date,usd_idr
0,2021-09-01,14284
1,2021-09-02,14281
2,2021-09-03,14261
3,2021-09-06,14239
4,2021-09-07,14195


In [3]:
news.head()

,published_at,date,clean_title,clean_content,dateline_location,word_count,sentence_count,paragraph_count,is_length_outlier,source_domain,language,url
0,2021-09-01 07:00:00,2021-09-01,"U.S. relationship with Taliban unclear after end of Afghanistan War, senior ...",Secretary of Defense Lloyd Austin said Wednesday it was not yet clear what k...,WASHINGTON,675,37,17,False,CNBC,en,https://www.cnbc.com/2021/09/01/afghanistan-update-us-relationship-with-tali...
1,2021-09-05 07:00:00,2021-09-05,"Taliban, opposition fight for Afghan holdout province, top U.S. general warn...",Taliban and opposition forces battled on Saturday to control the Panjshir Va...,NaN,914,41,32,False,CNBC,en,https://www.cnbc.com/2021/09/05/taliban-opposition-fight-for-afghan-holdout-...
2,2021-09-05 07:00:00,2021-09-05,"The Indian rupee has had a stable run this year, but UBS expects it will be ...",The Indian rupee has been one of the most stable currencies in Asia-Pacific ...,NaN,917,40,27,False,CNBC,en,https://www.cnbc.com/2021/09/06/indian-rupee-inr-outlook.html
3,2021-09-07 07:00:00,2021-09-07,"Taliban name new Afghan government, interior minister on U.S. sanctions list","The Taliban named Mullah Hasan Akhund, an associate of the movement's late f...",NaN,403,17,12,False,CNBC,en,https://www.cnbc.com/2021/09/07/taliban-name-new-afghan-government-amid-prot...
4,2021-09-09 07:00:00,2021-09-09,The U.S. has a record-breaking $1.73 trillion in student debt—borrowers from...,"In the United States, student debt has grown significantly over the past sev...",NaN,910,27,21,False,CNBC,en,https://www.cnbc.com/2021/09/09/america-has-1point73-trillion-in-student-deb...


## 3. Konversi Zona Waktu

Kolom `published_at` berasal dari `pubDate` RSS Google News, yang formatnya GMT (=UTC). Data kurs JISDOR mengikuti jam operasional pasar Indonesia (WIB, UTC+7). Supaya aturan "jam berapa market tutup" bisa diterapkan dengan benar, kedua data ini perlu disamakan zona waktunya dulu, kita konversi timestamp berita dari UTC ke WIB.

In [4]:
news["published_utc"] = news["published_at"].dt.tz_localize("UTC")
news["published_wib"] = news["published_utc"].dt.tz_convert("Asia/Jakarta")

news[["published_at", "published_wib"]].head()

,published_at,published_wib
0,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00
1,2021-09-05 07:00:00,2021-09-05 14:00:00+07:00
2,2021-09-05 07:00:00,2021-09-05 14:00:00+07:00
3,2021-09-07 07:00:00,2021-09-07 14:00:00+07:00
4,2021-09-09 07:00:00,2021-09-09 14:00:00+07:00


## 4. Aturan Penyelarasan Temporal

Aturan yang digunakan (roll-forward ke hari trading berikutnya):

1. Daftar "hari trading valid" diambil langsung dari tanggal-tanggal yang ada di data kurs JISDOR, jadi hari libur nasional maupun akhir pekan otomatis terdeteksi tanpa perlu kalender libur terpisah, karena BI memang tidak mempublikasikan kurs pada hari-hari tersebut.
2. Ditetapkan jam cutoff 16:00 WIB sebagai asumsi jam tutup pasar valas domestik. Berita yang terbit sebelum jam ini pada hari kerja dianggap mempengaruhi kurs hari itu juga.
3. Berita yang terbit **setelah** jam cutoff, atau pada akhir pekan/hari libur, digeser maju ke hari trading valid berikutnya.
4. Berita tidak dihapus, hanya dipetakan ulang ke tanggal yang paling masuk akal secara logis, supaya sinyalnya tetap terpakai untuk pengujian hipotesis di tahap selanjutnya.

In [5]:
CUTOFF_JAM = 16  # asumsi: pasar valas domestik dianggap tutup jam 16:00 WIB

trading_days = pd.DatetimeIndex(kurs["date"].dt.normalize().unique())


def geser_ke_trading_day(ts, trading_days=trading_days, cutoff=CUTOFF_JAM):
    """Memetakan satu timestamp berita ke hari trading valid berikutnya."""
    tanggal = ts.normalize().tz_localize(None)
    if ts.hour >= cutoff:
        tanggal += pd.Timedelta(days=1)
    while tanggal not in trading_days:
        tanggal += pd.Timedelta(days=1)
        if tanggal > trading_days.max() + pd.Timedelta(days=10):
            return pd.NaT  # di luar cakupan data kurs
    return tanggal


news["trading_day"] = news["published_wib"].apply(geser_ke_trading_day)

n_gagal = news["trading_day"].isna().sum()
print(f"Total berita         : {len(news)}")
print(f"Berhasil dipetakan   : {len(news) - n_gagal}")
print(f"Gagal dipetakan      : {n_gagal} (di luar rentang tanggal data kurs)")

Total berita         : 1719
Berhasil dipetakan   : 1700
Gagal dipetakan      : 19 (di luar rentang tanggal data kurs)


## 5. Agregasi Berita per Hari Trading

Karena satu hari trading bisa menampung lebih dari satu berita (dan sebaliknya, satu hari trading bisa tidak punya berita sama sekali karena strategi *sampling* bi-weekly pada tahap scraping), berita perlu diagregasi dulu menjadi satu baris per hari sebelum digabung ke data kurs. Di sini kita hitung jumlah berita dan menggabungkan judul-judulnya per hari.

In [7]:
agg = news.dropna(subset=["trading_day"]).groupby("trading_day").agg(
    jumlah_berita=("url", "count"),
    judul_gabungan=("clean_title", lambda x: " | ".join(x)),
).reset_index()

agg.head()

,trading_day,jumlah_berita,judul_gabungan
0,2021-09-01,1,"U.S. relationship with Taliban unclear after end of Afghanistan War, senior ..."
1,2021-09-06,2,"Taliban, opposition fight for Afghan holdout province, top U.S. general warn..."
2,2021-09-07,1,"Taliban name new Afghan government, interior minister on U.S. sanctions list"
3,2021-09-09,1,The U.S. has a record-breaking $1.73 trillion in student debt—borrowers from...
4,2021-09-10,2,Andy Card recalls one of the first calls President Bush made from Air Force ...


## 6. Penggabungan dengan Data Kurs 

In [8]:
final = kurs.merge(agg, left_on="date", right_on="trading_day", how="left")
final["jumlah_berita"] = final["jumlah_berita"].fillna(0).astype(int)
final["judul_gabungan"] = final["judul_gabungan"].fillna("")
final = final.drop(columns=["trading_day"])

final.head(10)

,date,usd_idr,jumlah_berita,judul_gabungan
0,2021-09-01,14284,1,"U.S. relationship with Taliban unclear after end of Afghanistan War, senior ..."
1,2021-09-02,14281,0,
2,2021-09-03,14261,0,
3,2021-09-06,14239,2,"Taliban, opposition fight for Afghan holdout province, top U.S. general warn..."
4,2021-09-07,14195,1,"Taliban name new Afghan government, interior minister on U.S. sanctions list"
5,2021-09-08,14266,0,
6,2021-09-09,14272,1,The U.S. has a record-breaking $1.73 trillion in student debt—borrowers from...
7,2021-09-10,14225,2,Andy Card recalls one of the first calls President Bush made from Air Force ...
8,2021-09-13,14260,5,The next fast-food chicken sandwich war may be a vegan one | ‘We were naive’...
9,2021-09-14,14257,0,


## 7. Validasi & Ringkasan Hasil

In [9]:
total_hari = len(final)
hari_ada_berita = (final["jumlah_berita"] > 0).sum()

print(f"Total hari trading                : {total_hari}")
print(f"Hari trading dengan >=1 berita     : {hari_ada_berita} ({hari_ada_berita/total_hari:.1%})")
print(f"Hari trading tanpa berita          : {total_hari - hari_ada_berita}")
print(f"Total berita ter-assign            : {final['jumlah_berita'].sum()}")
print(f"Berita di luar cakupan data kurs   : {n_gagal} dari {len(news)} ({n_gagal/len(news):.1%})")
print()

Total hari trading                : 1202
Hari trading dengan >=1 berita     : 857 (71.3%)
Hari trading tanpa berita          : 345
Total berita ter-assign            : 1700
Berita di luar cakupan data kurs   : 19 dari 1719 (1.1%)



berita 'di luar cakupan' terjadi karena tanggal scraping berita sedikit lebih baru dibanding tanggal terakhir data kurs yang tersedia 

## 8. Menyimpan Hasil Akhir 

In [10]:
OUTPUT_PATH = DATA_DIR / "processed" / "aligned_dataset.csv"
final.to_csv(OUTPUT_PATH, index=False)
print(f"Tersimpan di: {OUTPUT_PATH.resolve()}")
print(f"Bentuk akhir: {final.shape[0]} baris x {final.shape[1]} kolom")
final.dtypes

Tersimpan di: D:\Bismillah Kuliah\Semester 5\Pemrosesan Bahasan Alami\Tugas 1 - Kelompok\nlp-geopolitics-usa\data\processed\aligned_dataset.csv
Bentuk akhir: 1202 baris x 4 kolom


date              datetime64[ns]
usd_idr                    int64
jumlah_berita              int64
judul_gabungan            object
dtype: object